In [ ]:
import os
os.environ['ATTN_BACKEND'] = 'flash-attn'   # Can be 'flash-attn' or 'xformers', default is 'flash-attn'
os.environ['SPCONV_ALGO'] = 'auto'        # Can be 'native' or 'auto', default is 'auto'.
                                            # 'auto' is faster but will do benchmarking at the beginning.
                                            # Recommended to set to 'native' if run only once.
os.environ['OMP_NUM_THREADS'] = '4'        # Set the number of threads for OpenMP. Default is 4.
os.environ['TOKENIZERS_PARALLELISM'] = 'false' 
os.environ['USE_FLUX_DEV'] = 'true' 

import numpy as np
from matplotlib import pyplot as plt
import rembg
import gc
import torch
from PIL import Image
import imageio
from IPython.display import Video

# Import modular components
from modules.model_manager import get_model_manager
from modules.content_moderation import get_content_moderator  
from modules.gallery_manager import get_gallery_manager
from modules.generation_pipeline import get_generation_pipeline
import modules.print_pipeline as print_pipeline

# 🚀 CONFIGURATION SETUP 🚀
print("🔧 Configuring TRELLIS pipeline...")
device_config = {
    "flux": "cuda:0",     # FLUX with CPU offloading for max efficiency  
    "trellis": "cuda:0"   # TRELLIS on dedicated GPU (if available)
}

# Auto-detect best configuration based on available hardware
num_gpus = torch.cuda.device_count()
print(f"📊 Detected {num_gpus} GPU(s)")

if num_gpus >= 2 and device_config["flux"] != device_config["trellis"]:
    # Multi-GPU mode: TRELLIS CPU offloading will be automatically disabled
    print("🎮 Multi-GPU mode selected:")
    print(f"   FLUX: {device_config['flux']} (with CPU offloading)")
    print(f"   TRELLIS: {device_config['trellis']} (dedicated GPU, no CPU offloading)")
    print("   Note: Multi-GPU and TRELLIS CPU offloading are incompatible")
    model_manager = get_model_manager(device_config, enable_trellis_cpu_offload=True)  # Will be auto-disabled
else:
    # Single-GPU mode: Enable TRELLIS CPU offloading for memory efficiency
    device_config = {"flux": "cuda:0", "trellis": "cuda:0"}  # Force single GPU
    print("💾 Single-GPU mode selected:")
    print(f"   FLUX: {device_config['flux']} (with CPU offloading)")
    print(f"   TRELLIS: {device_config['trellis']} (with CPU offloading)")
    print("   Note: CPU offloading enabled for maximum memory efficiency")
    model_manager = get_model_manager(device_config, enable_trellis_cpu_offload=True)

content_moderator = get_content_moderator()
gallery_manager = get_gallery_manager()
generation_pipeline = get_generation_pipeline()

# Show final device configuration
print(f"\n📍 Final Configuration:")
print(f"  FLUX: {model_manager.flux_device} (CPU offloading: {model_manager.use_flux_cpu_offload})")
print(f"  TRELLIS: {model_manager.trellis_device} (CPU offloading: {model_manager.enable_trellis_cpu_offload})")

# Load all models
print("\n🚀 Loading models...")
flux_pipe, trellis_pipeline, reward_model = model_manager.load_all_models()

# Configure generation pipeline with models
generation_pipeline.set_models(
    flux_pipeline=flux_pipe,
    trellis_pipeline=trellis_pipeline,
    reward_model=reward_model,
    content_moderator=content_moderator
)

# Get generation config
config = model_manager.get_generation_config()
guidance_scale = config["guidance_scale"]
num_inference_steps = config["num_inference_steps"]

print("\n✅ All modules initialized and models loaded")
print(f"📊 Configuration: guidance_scale={guidance_scale}, num_inference_steps={num_inference_steps}")

# Check GPU memory usage
print(f"\n💾 GPU Memory Usage:")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {allocated:.1f}GB / {total:.1f}GB allocated")

In [ ]:
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
# model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m-it")
# messages = [
#     {"role": "user", "content": "Who are you?"},
# ]
# inputs = tokenizer.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
# ).to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=40)
# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
system_prompt = """You are a helpful assistant who creates prompts for generating rendered 3D objects. Focus on: \n 
- Avoid introductory phrases and meta-commentary, only write the prompt \n
- Add typical 3D rendering elements such as: Render, high quality 3D model, neutral background, optimized for 3D printing, ZBrush digital sculpt, stylized 3D model. \n
- Think about what the user wants and describe the desired object so that the generated image can resemble the 3D model of it. 
- Consider adding something like: safe-for-work, family-friendly if necessary"""

# system_prompt = """You are a helpful assistant who creates prompts to generate images of rendered 3D objects. Avoid introductory phrases and meta-commentary."""

def create_image_prompt(start_prompt):
    """Generate a summary for a single caption using the Gemma model"""
    # Load and process the image
    
    messages = [
                {
            "role": "system",
           "content": [{"type": "text", "text": system_prompt}]
        },
    {
        "role": "user",
        "content": [

            {"type": "text", "text":"Create a prompt based on: " + start_prompt}
        ]
    }
    ]
    return messages

In [ ]:
# Define prompt and generation parameters
prompt = "a pretzel-copter, a helicopter where the cockpit is in a pretzel shape and the whole helicopter looks like a pretzel"

enhanced_prompt_msg = create_image_prompt(prompt)
enhanced_prompt_inputs = tokenizer.apply_chat_template(
	enhanced_prompt_msg,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

enhanced_prompt_outputs = model.generate(**enhanced_prompt_inputs, max_new_tokens=100)
print(tokenizer.decode(enhanced_prompt_outputs[0][enhanced_prompt_inputs["input_ids"].shape[-1]:]))

In [ ]:
prompt = tokenizer.decode(enhanced_prompt_outputs[0][enhanced_prompt_inputs["input_ids"].shape[-1]:])
print(prompt)

In [ ]:
# Get next print number using gallery manager
file_number = gallery_manager.get_next_print_number()
print(f"▶️ Starting generation with FILE_NUMBER: {file_number}")

# Define prompt and generation parameters
# prompt = "potato anime women"

batch_size = 4

# Check text safety first
is_safe, scores = content_moderator.check_text_safety(prompt)
if not is_safe:
    print(f"⚠️ Prompt flagged by content moderation: {scores}")
    print("Please use a different prompt.")
else:
    print("✅ Prompt passed content moderation")
    
    # Use generation pipeline to create images
    print("🎨 Generating images...")
    filtered_images, images = generation_pipeline.generate_images_return_raw(
        prompt, 
        num_images=batch_size,
        base_seed=13,  # Fixed seed for reproducibility
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps
    )
    
    print(f"✅ Generated {len(filtered_images)} images")
    
    # Display images with background removal
    plt.figure(figsize=(20, 15))
    for i in range(len(filtered_images)):
        plt.subplot(1, len(filtered_images), i + 1)
        image = filtered_images[i]
        output = rembg.remove(image)
        plt.imshow(output)
        plt.axis('off')
    plt.show()

In [ ]:
plt.figure(figsize=(20, 15))
for i in range(len(images)):
    plt.subplot(1, len(images), i + 1)
    image = images[i]
    output = rembg.remove(image)
    plt.imshow(output)
    plt.axis('off')
plt.show()

In [ ]:
# # Score images with reward model (already loaded through model manager)
# import hpsv2

# # Get the refined prompt used by the generation pipeline
# refined_prompt = generation_pipeline._create_enhanced_prompt(prompt)
# print(f"Original prompt: {prompt}")
# print(f"Refined prompt: {refined_prompt}")

# # Score with HPS v2
# result = hpsv2.score(filtered_images, refined_prompt, hps_version="v2.0") 
# print(f"HPS v2 scores: {result}")

# # Score with reward model
# rewards = reward_model.score(prompt, filtered_images)
# print(f"Reward model scores: {rewards}")

# # Select best image
# best_image_idx = np.argmax(rewards)
# selected_image = filtered_images[best_image_idx]
# print(f"Selected image {best_image_idx} with reward {rewards[best_image_idx]:.4f}")
best_image_idx = 1
selected_image = filtered_images[best_image_idx]
selected_image

In [ ]:
# # Clear memory before 3D generation
# gc.collect()
# torch.cuda.empty_cache()

print("🔮 Generating 3D model...")

# Use generation pipeline for 3D model creation with CPU offloading and simple GLB export
video_path, glb_path = generation_pipeline.generate_3d_model(
    selected_image,
    sample_video=True,
    texture_size=512,
    base_seed=3,  # Fixed seed for reproducibility
    use_simple_glb=False  # Use memory-efficient GLB export to avoid 66TB bug
)

print(f"✅ 3D model generation complete!")
print(f"📹 Video saved to: {video_path}")
print(f"📦 GLB model saved to: {glb_path}")

In [ ]:
# Prepare model for 3D printing using the print pipeline
print(f"🖨️ Preparing model for 3D printing with number: {file_number}")

# Run the print pipeline to create STL file
stl_filepath = print_pipeline.run_with_file(glb_path, file_number, output_folder=gallery_manager.output_dir)
print(f"✅ STL file created: {stl_filepath}")

# Save to gallery using gallery manager
gallery_manager.save_item(
    file_number,
    prompt,  # Original prompt
    selected_image,
    stl_filepath,
    glb_path, 
    video_path
)

# Increment counter for next generation
gallery_manager.increment_counter()

print(f"✅ Item saved to gallery with print number: {file_number}")
print(f"📊 Gallery stats: {gallery_manager.get_stats()}")

In [ ]:
# if len(video_path) > 0:
Video(video_path, embed=True, width=800)